In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)

(7043, 21)


In [2]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

Duplicate rows: 0
Duplicate customer IDs: 0


In [3]:
df["TotalCharges"].dtype

<StringDtype(na_value=nan)>

In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [5]:
df["TotalCharges"].isnull().sum()

np.int64(11)

In [6]:
df["TotalCharges"].isnull().sum()

np.int64(11)

In [7]:
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


In [8]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["TotalCharges"].isnull().sum()

np.int64(0)

In [9]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

In [10]:
df["OnlineSecurity"].value_counts()

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

In [11]:
cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies"]

for c in cols:
    df[c] = df[c].replace("No internet service", "No")

df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

In [12]:
df["OnlineSecurity"].value_counts()


OnlineSecurity
No     5024
Yes    2019
Name: count, dtype: int64

In [13]:
# Churn as a number (1 = left, 0 = stayed). Models need numbers.

df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})

# Group tenure into bands for segmentation
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
)

In [14]:
df[["tenure", "TenureGroup", "Churn", "ChurnFlag"]].head(10)

,tenure,TenureGroup,Churn,ChurnFlag
0,1,0-12 months,No,0
1,34,25-48 months,No,0
2,2,0-12 months,Yes,1
3,45,25-48 months,No,0
4,2,0-12 months,Yes,1
5,8,0-12 months,Yes,1
6,22,13-24 months,No,0
7,10,0-12 months,No,0
8,28,25-48 months,Yes,1
9,62,49-72 months,No,0


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customerID        7043 non-null   str     
 1   gender            7043 non-null   str     
 2   SeniorCitizen     7043 non-null   str     
 3   Partner           7043 non-null   str     
 4   Dependents        7043 non-null   str     
 5   tenure            7043 non-null   int64   
 6   PhoneService      7043 non-null   str     
 7   MultipleLines     7043 non-null   str     
 8   InternetService   7043 non-null   str     
 9   OnlineSecurity    7043 non-null   str     
 10  OnlineBackup      7043 non-null   str     
 11  DeviceProtection  7043 non-null   str     
 12  TechSupport       7043 non-null   str     
 13  StreamingTV       7043 non-null   str     
 14  StreamingMovies   7043 non-null   str     
 15  Contract          7043 non-null   str     
 16  PaperlessBilling  7043 non-null   s

In [16]:
df["TenureGroup"] = df["TenureGroup"].astype(str)
df.to_csv("../data/telco_clean.csv", index=False)
print("Saved!", df.shape)

Saved! (7043, 23)


In [17]:
df.groupby("TenureGroup")["ChurnFlag"].mean().mul(100).round(1)

TenureGroup
0-12 months     47.4
13-24 months    28.7
25-48 months    20.4
49-72 months     9.5
Name: ChurnFlag, dtype: float64

In [18]:
churned = df[df["ChurnFlag"] == 1]
churned["TenureGroup"].value_counts(normalize=True).mul(100).round(1)

TenureGroup
0-12 months     55.5
25-48 months    17.4
13-24 months    15.7
49-72 months    11.4
Name: proportion, dtype: float64

In [19]:
churned = df[df["ChurnFlag"] == 1]
churned["TenureGroup"].value_counts(normalize=True).mul(100).round(1)

TenureGroup
0-12 months     55.5
25-48 months    17.4
13-24 months    15.7
49-72 months    11.4
Name: proportion, dtype: float64